In [ ]:
import os
from PIL import Image
from torch.utils.data import Dataset
import numpy as np

import albumentations as A
from albumentations.pytorch import ToTensorV2
# from tgdm import tgdm
import torch.nn as nn 
import torch.optim as optim

In [ ]:
# Hyperparameters
IMAGE_HEIGHT = 160
IMAGE_WIDTH = 240

In [ ]:
class CovidDataset(Dataset):
  __prefix = '.\covid-segmentation'
  
  def __init__(self, images_path, mask_path, transform=None):
    self.images_path = images_path
    self.mask_path = mask_path
    self.transform = transform
    self.images = os.path.join(self.__prefix, images_path)
    
  def __len__(self):
    return len(self.images)
    
  def __getitem__(self, index):
    img_path = os.path.join(self.image_path, self.images[index])
    mask_path = os.path.join(self.mask_path, self.images[index].replace(".jpg", "_mask.gif"))
    image = np.array(Image.open(img_path).convert("RGB"))
    mask = np.array(Image.open(mask_path).convert("L"), dtype=np.float32)
    mask[mask == 255.0] = 1.0
    
    if self.transform is not None:
      augmentations = self.transform(image=image, mask=mask)
      image = augmentations["image"]
      mask = augmentations["mask"]
      return image, mask
    

In [ ]:
class Transformer():
  def __init__(self, img_height=160, img_width=240):
    self.img_height = img_height
    self.img_width = img_width
    
  def pipeline(self, train=True):
    if train == True:
      return A.Compose(
        [
          A.Resize(height=self.height, width=self.width),
          A.Rotate(limit=35, p=1.0),
        ]
      )
      # [
      #           A.Resize(height=self.height, width=self.width),
      #           A.Rotate(limit=35, p=1.0),
      #           A.HorizontalFlip=(p=0.5),
      #           A.VerticalFlip(p=0.1),
      #           A.Normalize(
      #             mean=[0.0, 0.0, 0.0],
      #             std=[1.0, 1.0, 1.0],
      #             max_pixel_value=255.0
      #           ),
      #           ToTensorV2()
      #         ]
    else:
      return A.Compose(
        [
          A.Resize(height=self.img_height, width=self.img_width),
          A.Normalize(
            mean=[0.0, 0.0, 0.0],
            std=[1.0, 1.0, 1.0],
            max_pixl_value=255.0
          ),
          ToTensorV2()
        ]
      )
    